# Multimodal Emotion Recognition — Full Training Notebook
**Thesis: Amal Omanakuttan | TU Dublin (DIT)**

## What must be in `My Drive/Thesis/` before running

| File / Folder | Status | Notes |
|---|---|---|
| `DATASET/train/1..7/` | ✅ Already uploaded | All your face images |
| `DATASET/test/1..7/` | ✅ Already uploaded | Test images |
| `src/` | ✅ Already uploaded | Python source code |
| `audio_data.zip` | Upload now | Zip of the audio_data folder |

**You do NOT need to upload CSVs or any checkpoint — this notebook generates everything from scratch.**

## Training plan (~2.5 hours on T4 GPU)
| Cell | What it does | Time |
|---|---|---|
| 1–4 | GPU check, Drive mount, install deps, copy files | ~5 min |
| 5 | Auto-generate CSVs from your actual Drive images | instant |
| 6 | Verify dataset — shows image count per class | instant |
| 7 | **Train face model (DINO ViT-S/16)** | ~60–90 min |
| 8 | **Train audio model (Wav2Vec2)** | ~30–45 min |
| 9 | **Train fusion model** | ~10 min |
| 10 | Final report — accuracy + F1 for all 3 models | instant |

**Steps: Runtime → Change runtime type → T4 GPU → Save → Runtime → Run all**

In [ ]:
# Cell 1 -- Check GPU
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('NO GPU! Go to Runtime -> Change runtime type -> T4 GPU -> Save')

In [ ]:
# Cell 2 -- Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/Thesis'
assert os.path.isdir(DRIVE_ROOT), f'Cannot find {DRIVE_ROOT} -- did Drive mount correctly?'
print('Drive mounted. Contents:', os.listdir(DRIVE_ROOT))

In [ ]:
# Cell 3 -- Install dependencies
!pip install -q 'transformers>=4.40.0' Pillow soundfile torchaudio librosa

In [ ]:
# Cell 4 -- Set up working directory, copy files from Drive
import os, shutil, zipfile, sys

WORK_DIR   = '/content/thesis'  # DRIVE_ROOT already set in Cell 2
os.makedirs(f'{WORK_DIR}/checkpoints', exist_ok=True)

# --- Copy source code ---
shutil.copytree(f'{DRIVE_ROOT}/src', f'{WORK_DIR}/src', dirs_exist_ok=True)
sys.path.insert(0, f'{WORK_DIR}/src')
print('Source code copied')

def extract_zip_safe(zip_path, extract_dir):
    """Extract zip and fix Windows backslashes in paths (Compress-Archive creates these)."""
    with zipfile.ZipFile(zip_path, 'r') as zf:
        for member in zf.namelist():
            corrected = member.replace('\\', '/')
            target = os.path.join(extract_dir, corrected)
            if corrected.endswith('/'):
                os.makedirs(target, exist_ok=True)
            else:
                os.makedirs(os.path.dirname(target), exist_ok=True)
                with zf.open(member) as src, open(target, 'wb') as dst:
                    shutil.copyfileobj(src, dst)

# --- Extract DATASET ---
dataset_zip = f'{DRIVE_ROOT}/DATASET.zip'
assert os.path.exists(dataset_zip), 'DATASET.zip not found in Drive!'
print('Extracting DATASET.zip ...')
extract_zip_safe(dataset_zip, WORK_DIR)
print('DATASET extracted')

# --- Extract audio ---
audio_zip = f'{DRIVE_ROOT}/audio_data.zip'
assert os.path.exists(audio_zip), 'audio_data.zip not found in Drive!'
print('Extracting audio_data.zip ...')
extract_zip_safe(audio_zip, WORK_DIR)
print('Audio data extracted')

# Verify structure
train_classes = sorted(os.listdir(f'{WORK_DIR}/DATASET/train'))
test_classes  = sorted(os.listdir(f'{WORK_DIR}/DATASET/test'))
print(f'Train class folders: {train_classes}')
print(f'Test  class folders: {test_classes}')
audio_actors  = sorted(os.listdir(f'{WORK_DIR}/audio_data'))[:5]
print(f'Audio (first 5): {audio_actors}')


In [ ]:
# Cell 5 -- Auto-generate train_labels.csv and test_labels.csv
# This reads the actual files on disk so it works with ANY number of images,
# including any extra images you added beyond the original RAF-DB.
import csv, os

VALID_EXT = {'.jpg', '.jpeg', '.png'}

def generate_csv(img_root, out_csv):
    rows = []
    for cls_folder in sorted(os.listdir(img_root)):
        cls_path = os.path.join(img_root, cls_folder)
        if not os.path.isdir(cls_path):
            continue
        try:
            label = int(cls_folder)  # folder name IS the label (1-7)
        except ValueError:
            continue
        for fname in sorted(os.listdir(cls_path)):
            if os.path.splitext(fname)[1].lower() in VALID_EXT:
                rows.append({'image': fname, 'label': label})
    with open(out_csv, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['image', 'label'])
        writer.writeheader()
        writer.writerows(rows)
    return len(rows)

EMOTION = {1:'Surprise', 2:'Fear', 3:'Disgust', 4:'Happy', 5:'Sad', 6:'Anger', 7:'Neutral'}

n_train = generate_csv(f'{WORK_DIR}/DATASET/train', f'{WORK_DIR}/train_labels.csv')
n_test  = generate_csv(f'{WORK_DIR}/DATASET/test',  f'{WORK_DIR}/test_labels.csv')

print(f'train_labels.csv generated: {n_train:,} images')
print(f'test_labels.csv  generated: {n_test:,}  images')

# Show per-class breakdown
from collections import Counter
import csv as _csv
def count_csv(path):
    with open(path) as f:
        return Counter(int(r['label']) for r in _csv.DictReader(f))

print('\nTrain distribution:')
for cls, n in sorted(count_csv(f'{WORK_DIR}/train_labels.csv').items()):
    bar = '#' * (n * 30 // n_train)
    print(f'  {EMOTION[cls]:10s} [{bar:<30s}] {n:5d}')

print('\nTest distribution:')
for cls, n in sorted(count_csv(f'{WORK_DIR}/test_labels.csv').items()):
    bar = '#' * (n * 30 // n_test)
    print(f'  {EMOTION[cls]:10s} [{bar:<30s}] {n:5d}')

In [ ]:
# Cell 6 -- Verify DataLoaders load correctly
from dataset import build_dataloaders, LABEL_MAP
from models  import DINOFERModel, count_trainable_params
import torch

device = torch.device('cuda')
EMOTION_NAMES = [LABEL_MAP[i + 1] for i in range(7)]

train_loader, val_loader, test_loader = build_dataloaders(
    csv_path      = f'{WORK_DIR}/train_labels.csv',
    train_img_dir = f'{WORK_DIR}/DATASET/train',
    test_dir      = f'{WORK_DIR}/DATASET/test',
    batch_size    = 32,
    num_workers   = 2,
    image_size    = 224,
    val_split     = 0.1,
)

imgs, lbls = next(iter(train_loader))
print(f'Batch shape: {tuple(imgs.shape)}  Labels: {lbls[:8].tolist()}')
print('DataLoaders OK')

In [ ]:
# Cell 7 -- Train Face Model (DINO ViT-S/16)  ~60-90 min on T4
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import warnings

EPOCHS    = 20
LR        = 5e-5
GRAD_CLIP = 1.0

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    face_model = DINOFERModel(
        model_variant          = 'vits16',
        freeze                 = False,
        unfreeze_last_n_blocks = 4,
    ).to(device)

print(f'Trainable params: {count_trainable_params(face_model):,}')

criterion  = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer  = AdamW(filter(lambda p: p.requires_grad, face_model.parameters()),
                   lr=LR, weight_decay=0.01)
scheduler  = CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_loader), eta_min=1e-6)

def batch_acc(logits, labels):
    return (logits.argmax(1) == labels).float().mean().item()

best_face_val_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    face_model.train()
    t_loss = t_acc = n = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = face_model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(face_model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        t_loss += loss.item(); t_acc += batch_acc(logits, labels); n += 1

    face_model.eval()
    v_loss = v_acc = v_n = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = face_model(imgs)
            v_loss += criterion(logits, labels).item()
            v_acc  += batch_acc(logits, labels)
            v_n    += 1

    val_acc = v_acc / v_n
    history['train_loss'].append(t_loss / n)
    history['train_acc'].append(t_acc / n)
    history['val_loss'].append(v_loss / v_n)
    history['val_acc'].append(val_acc)
    print(f'Epoch {epoch:02d}/{EPOCHS}  '
          f'train_loss={t_loss/n:.4f}  train_acc={t_acc/n:.4f}  '
          f'val_loss={v_loss/v_n:.4f}  val_acc={val_acc:.4f}')

    if val_acc > best_face_val_acc:
        best_face_val_acc = val_acc
        torch.save(face_model.state_dict(), f'{WORK_DIR}/checkpoints/best_dino.pt')
        torch.save(face_model.state_dict(), f'{DRIVE_ROOT}/best_dino.pt')  # backup to Drive
        print(f'  [BEST] val_acc={best_face_val_acc:.4f} -- saved to Drive')

print(f'\nFace training done. Best val acc: {best_face_val_acc:.4f}')

# --- Test set evaluation ---
face_model.load_state_dict(torch.load(f'{WORK_DIR}/checkpoints/best_dino.pt', map_location=device))
face_model.eval()
all_preds, all_labels_list = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        preds = face_model(imgs.to(device)).argmax(1).cpu().tolist()
        all_preds.extend(preds)
        all_labels_list.extend(labels.tolist())

face_test_acc = sum(p == l for p, l in zip(all_preds, all_labels_list)) / len(all_labels_list)
tp = [0]*7; fp = [0]*7; fn = [0]*7
for p, l in zip(all_preds, all_labels_list):
    if p == l: tp[l] += 1
    else:      fp[p] += 1; fn[l] += 1
face_f1s = [2*tp[c]/(2*tp[c]+fp[c]+fn[c]) if (2*tp[c]+fp[c]+fn[c])>0 else 0 for c in range(7)]
face_macro_f1 = sum(face_f1s) / 7

print(f'\nFace Model Test Accuracy : {face_test_acc*100:.2f}%')
print(f'Face Model Macro F1      : {face_macro_f1*100:.2f}%')
print('Per-class:')
from collections import Counter
correct = Counter(l for p, l in zip(all_preds, all_labels_list) if p == l)
total   = Counter(all_labels_list)
for i, name in enumerate(EMOTION_NAMES):
    a = correct[i] / max(total[i], 1)
    print(f'  {name:10s} {a*100:.1f}%  (F1={face_f1s[i]*100:.1f}%)')


import json as _json
face_results = {
    'model': 'dino',
    'test_acc': round(face_test_acc, 4),
    'macro_f1': round(face_macro_f1, 4),
    'best_val_acc': round(best_face_val_acc, 4),
    'per_class_acc': {name: round(correct[i] / max(total[i], 1), 4)
                      for i, name in enumerate(EMOTION_NAMES)},
    'history': history,
}
with open(f'{WORK_DIR}/checkpoints/results_dino.json', 'w') as f:
    _json.dump(face_results, f, indent=2)
shutil.copy(f'{WORK_DIR}/checkpoints/results_dino.json', f'{DRIVE_ROOT}/results_dino.json')
print('results_dino.json saved (with real per-epoch history) and backed up to Drive')


In [ ]:
# Cell 8 -- Train Audio Model (Wav2Vec2)  ~45-60 min on T4
# --unfreeze: trains the Wav2Vec2 backbone (not just the head)
# This takes longer but gives much better accuracy than frozen backbone (20% -> ~50-60%)
import os
os.chdir(WORK_DIR)

!python src/train_audio.py \
    --data_root  audio_data \
    --dataset    ravdess \
    --epochs     20 \
    --batch_size 8 \
    --lr         5e-5 \
    --unfreeze \
    --output_dir checkpoints

import shutil
shutil.copy('checkpoints/best_audio.pt', f'{DRIVE_ROOT}/best_audio.pt')
print('\nbest_audio.pt saved to Google Drive')


In [ ]:
# Cell 9 -- Train Fusion Model (face + audio late fusion)  ~10 min on T4
# Synthetic pairing: randomly pairs face+audio from the same emotion class.
# n_synthetic=4000 means 4000 training pairs are generated from your face+audio data.

!python src/train_multimodal.py \
    --face_ckpt  checkpoints/best_dino.pt \
    --audio_ckpt checkpoints/best_audio.pt \
    --synthetic \
    --img_root   DATASET/train \
    --img_csv    train_labels.csv \
    --audio_root audio_data \
    --n_synthetic 4000 \
    --epochs     20 \
    --output_dir checkpoints

import shutil
shutil.copy('checkpoints/best_fusion.pt', f'{DRIVE_ROOT}/best_fusion.pt')
print('\nbest_fusion.pt saved to Google Drive')

In [ ]:
# Cell 10 -- Final Report
import json, os

print('=' * 60)
print('  FINAL RESULTS SUMMARY')
print('=' * 60)

print(f'\n[1] Face Model (DINO ViT-S/16) -- trained on YOUR dataset')
print(f'    Test Accuracy : {face_test_acc*100:.2f}%')
print(f'    Macro F1      : {face_macro_f1*100:.2f}%')
print(f'    Baseline      : random=14.3%  majority-class~38%')

for label, fname in [('[2] Audio Model (Wav2Vec2)', 'results_audio.json'),
                     ('[3] Fusion Model (Face+Audio)', 'results_fusion.json')]:
    path = f'checkpoints/{fname}'
    if os.path.exists(path):
        with open(path) as f:
            r = json.load(f)
        print(f'\n{label}')
        print(f'    Best Val Accuracy : {r.get("best_val_acc", r.get("val_acc", "?"))}')
        if 'macro_f1' in r:
            print(f'    Macro F1          : {r["macro_f1"]}')
    else:
        print(f'\n{label} -- results file not found (training may have failed)')

print('\n' + '=' * 60)
print('Checkpoints on Google Drive:')  # DRIVE_ROOT already set in Cell 2
for f in ['best_dino.pt', 'best_audio.pt', 'best_fusion.pt']:
    path = f'{DRIVE_ROOT}/{f}'
    size = os.path.getsize(path) // (1024*1024) if os.path.exists(path) else 0
    status = f'OK ({size} MB)' if os.path.exists(path) else 'MISSING'
    print(f'  {status:15s} {f}')

print('\nNext steps:')
print('  1. Download best_dino.pt, best_audio.pt, best_fusion.pt from Drive')
print('  2. Put all three in checkpoints/ on your laptop')
print('  3. Run: .\\venv\\Scripts\\python.exe demo_professor.py')